# 1. Obesity Data Preprocessing

**Goal:** Load the raw dataset, perform exploratory analysis, and save reusable preprocessing artifacts for downstream notebooks.

**Core design rule: prevent data leakage**
- `Height` and `Weight` are direct upstream variables of the target via BMI.
- Using them as features lets the model relearn the obesity label definition.
- This rewritten pipeline uses **behavioral and lifestyle features only**.

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

DATA_PATH = '../data/obesity.csv'
ARTIFACTS_DIR = '../artifacts'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

In [ ]:
data = pd.read_csv(DATA_PATH)
print(f'Shape: {data.shape}')
print(f'Columns: {list(data.columns)}')
data.head()

## Target distribution

In [ ]:
obesity_order = [
    'Insufficient_Weight', 'Normal_Weight',
    'Overweight_Level_I', 'Overweight_Level_II',
    'Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III'
]
counts = data['NObeyesdad'].value_counts().reindex(obesity_order)
plt.figure(figsize=(10, 4))
sns.barplot(x=obesity_order, y=counts.values, palette='RdYlGn_r')
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.title('Obesity Label Distribution')
plt.ylabel('Count')
plt.tight_layout()
plt.show()
print(counts)

## Feature selection

This project keeps only behavioral and lifestyle predictors and excludes `Height` and `Weight`.

In [ ]:
CATEGORICAL_FEATURES = [
    'Gender', 'CALC', 'FAVC', 'SCC', 'SMOKE',
    'family_history_with_overweight', 'CAEC', 'MTRANS'
]
CONTINUOUS_FEATURES = ['Age', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
TARGET = 'NObeyesdad'

print('Selected feature groups:')
print(f'  Categorical: {CATEGORICAL_FEATURES}')
print(f'  Continuous: {CONTINUOUS_FEATURES}')
print('  Excluded due to leakage: [Height, Weight]')

## EDA: continuous feature distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for axis, column in zip(axes.flatten(), CONTINUOUS_FEATURES):
    sns.histplot(data[column], kde=True, ax=axis, bins=25, color='steelblue')
    axis.set_title(column)
plt.suptitle('Behavioral Continuous Features', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## EDA: categorical feature distributions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for axis, column in zip(axes.flatten(), CATEGORICAL_FEATURES):
    value_counts = data[column].value_counts()
    sns.barplot(x=value_counts.index, y=value_counts.values, ax=axis, palette='Set2')
    axis.set_title(column)
    axis.tick_params(axis='x', rotation=30)
plt.suptitle('Categorical Feature Distributions', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Encode categorical variables

- Binary features use fixed dictionary mappings
- Ordered features use explicit ordinal mappings
- The mappings are stored in `artifacts/encoders.pkl` for reuse during inference

In [ ]:
BINARY_MAP = {
    'Gender': {'Female': 0, 'Male': 1},
    'FAVC': {'no': 0, 'yes': 1},
    'SCC': {'no': 0, 'yes': 1},
    'SMOKE': {'no': 0, 'yes': 1},
    'family_history_with_overweight': {'no': 0, 'yes': 1},
}
ORDINAL_MAP = {
    'CALC': ['no', 'Sometimes', 'Frequently', 'Always'],
    'CAEC': ['no', 'Sometimes', 'Frequently', 'Always'],
    'MTRANS': ['Bike', 'Walking', 'Public_Transportation', 'Motorbike', 'Automobile'],
}

processed = data.copy()
for column, mapping in BINARY_MAP.items():
    processed[column] = processed[column].map(mapping)
for column, order in ORDINAL_MAP.items():
    processed[column] = processed[column].map({value: index for index, value in enumerate(order)})

encoders = {'binary': BINARY_MAP, 'ordinal': ORDINAL_MAP}
with open(f'{ARTIFACTS_DIR}/encoders.pkl', 'wb') as file:
    pickle.dump(encoders, file)

print('Saved artifacts/encoders.pkl')
processed[CATEGORICAL_FEATURES].head()

## Scale continuous features

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
processed[CONTINUOUS_FEATURES] = scaler.fit_transform(processed[CONTINUOUS_FEATURES])
with open(f'{ARTIFACTS_DIR}/scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

print('Saved artifacts/scaler.pkl')
print('Scaled feature means:')
print(processed[CONTINUOUS_FEATURES].mean().round(4).to_dict())

## Correlation analysis

In [ ]:
feature_frame = processed[CATEGORICAL_FEATURES + CONTINUOUS_FEATURES]
correlation_matrix = feature_frame.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.4)
plt.title('Behavioral Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## Save the processed dataset

In [ ]:
save_columns = CATEGORICAL_FEATURES + CONTINUOUS_FEATURES + [TARGET]
processed[save_columns].to_csv(f'{ARTIFACTS_DIR}/processed_data.csv', index=False)
print(f'Saved artifacts/processed_data.csv with shape {processed[save_columns].shape}')
processed[save_columns].head()